### LangChain

In [57]:
# Run this once in the notebook to install the required packages.
# %pip install -U langchain langchain-openai langchain-text-splitters

from pathlib import Path
import httpx2
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from rich import print
from pydantic import SecretStr
from langchain.agents import create_agent
from typing import List
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter, MarkdownHeaderTextSplitter

In [58]:
def print_conversation(messages:List[BaseMessage]) -> None:
    for message in messages:
       message.pretty_print() 


In [59]:
openai_api_key = SecretStr(Path('openai-secret-key-ai-integrations-developers.txt').read_text(encoding='utf-8').strip())
openai_model = ChatOpenAI(
    model_name='gpt-5-nano',
    openai_api_key=openai_api_key,
    reasoning_effort='low',
    http_client=httpx2.Client(trust_env=False),
)

### Short-term Memory

In [60]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver keeps each thread's messages only while this notebook kernel runs.
memory = InMemorySaver()
agent = create_agent(
    model=openai_model,
    tools=[],
    system_prompt="You are a helpful assistant.",
    checkpointer=memory,
)

# This ID identifies one separate conversation.
thread_1 = {"configurable": {"thread_id": "thread_1"}}


In [61]:
# First message in thread_1: the checkpointer saves it.
result_1 = agent.invoke(
    {"messages": [HumanMessage(content="Hello, AI! My name is Maria.")]},
    config=thread_1,
)
print(result_1["messages"][-1].content)


Hi Maria! Nice to meet you. How can I help today? If you have a question, need a quick answer, or just want to 
chat, I’m here.

In [62]:
# Same thread ID: the agent receives the earlier message automatically.
result_2 = agent.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=thread_1,
)
print(result_2["messages"][-1].content)  # The answer should be Tony.


Your name is Maria. How can I assist you today, Maria?

### Long-term Memory

In [63]:
from langgraph.store.memory import InMemoryStore

In [64]:
# Create these once per kernel session. Do not rerun this cell between saving and recalling facts.
store = InMemoryStore()
checkpointer = InMemorySaver()

### Long-term memory tools

In [65]:
# Pydantic on Python 3.11 requires TypedDict from typing_extensions.
from typing_extensions import TypedDict
from langchain.tools import ToolRuntime, tool


# The agent receives this value at invocation time. It keeps each user's facts separate.
class CustomAgentContext(TypedDict):
    user_id: str


@tool
def remember_user_facts(
    key: str,
    value: str,
    runtime: ToolRuntime[CustomAgentContext],
) -> str:
    """Extract durable user facts from a user message and store them in long-term memory.

    Example: key='allergy', value='The user is allergic to nuts.'

    Args:
        key: A unique identifier for the fact.
        value: The fact itself.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    previous_item = runtime.store.get(namespace, "auto_extracted_facts")

    facts_dict = previous_item.value if previous_item is not None else {}
    facts_dict[key] = value

    runtime.store.put(namespace, "auto_extracted_facts", facts_dict)
    return "OK"


@tool
def recall_user_facts(runtime: ToolRuntime[CustomAgentContext]) -> str:
    """Recall previously stored long-term facts about the user."""
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    results = runtime.store.search(namespace, limit=20)

    if not results:
        return "No facts stored."

    return "\n---\n".join(
        f"{facts_group.key}:\n"
        + "\n".join(
            f"- {key}: {value}" for key, value in facts_group.value.items()
        )
        for facts_group in results
    )


### Agent with long-term user facts

In [66]:
from langchain_core.runnables import RunnableLambda


# Reuse the one store and checkpointer created above.

long_term_memory_agent = create_agent(
    model=openai_model,
    tools=[remember_user_facts, recall_user_facts],
    system_prompt=f"""
You are a polite and helpful personal assistant.
At the start of every interaction, call `{recall_user_facts.name}` to check
whether facts about this user have already been stored.
When the user provides a durable fact (for example a name, hobby, plan,
need, preference, or location), call `{remember_user_facts.name}` after
recalling and before your final answer. Do this even when the user does not
explicitly say the word 'remember'.
Be friendly and use known facts when they are relevant.
""",
    checkpointer=checkpointer,
    store=store,
    context_schema=CustomAgentContext,
)

# This runnable prints the complete message history after each agent invocation.
interact = long_term_memory_agent | RunnableLambda(
    lambda result: print_conversation(result["messages"])
)


In [67]:
# This call recalls first, then saves the name, city, and hobby in `store`.
save_result = long_term_memory_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Maria, I live in Sofia, and I enjoy hiking."
            )
        ]
    },
    config={"configurable": {"thread_id": "maria-thread-1"}},
    context={"user_id": "maria"},
)
print_conversation(save_result["messages"])


================================ Human Message =================================

My name is Maria, I live in Sofia, and I enjoy hiking.
================================== Ai Message ==================================
Tool Calls:
  recall_user_facts (call_uZzv3uYaj4SPgkb7hB8FI7Lq)
 Call ID: call_uZzv3uYaj4SPgkb7hB8FI7Lq
  Args:
================================= Tool Message =================================
Name: recall_user_facts

No facts stored.
================================== Ai Message ==================================
Tool Calls:
  remember_user_facts (call_iZFtdXWH2Rl1gsRBWBl8HQRb)
 Call ID: call_iZFtdXWH2Rl1gsRBWBl8HQRb
  Args:
    key: name
    value: Maria
================================= Tool Message =================================
Name: remember_user_facts

OK
================================== Ai Message ==================================
Tool Calls:
  remember_user_facts (call_WfPrdr7pbrG1GFc3hlaEAtEf)
 Call ID: call_WfPrdr7pbrG1GFc3hlaEAtEf
  Args:
    key: locati

In [68]:
# A new thread has no chat history, but the same user_id reads facts from the same store.
recall_result = long_term_memory_agent.invoke(
    {"messages": [HumanMessage(content="What do you remember about me?")]},
    config={"configurable": {"thread_id": "maria-thread-2"}},
    context={"user_id": "maria"},
)
print_conversation(recall_result["messages"])


================================ Human Message =================================

What do you remember about me?
================================== Ai Message ==================================
Tool Calls:
  recall_user_facts (call_HTSLIRDJBzUvHjdHWXHJ5JoG)
 Call ID: call_HTSLIRDJBzUvHjdHWXHJ5JoG
  Args:
================================= Tool Message =================================
Name: recall_user_facts

auto_extracted_facts:
- name: Maria
- location: Sofia
- hobby: hiking
================================== Ai Message ==================================
Tool Calls:
  remember_user_facts (call_EsNciBeRze1YwsEa93ThqsUl)
 Call ID: call_EsNciBeRze1YwsEa93ThqsUl
  Args:
    key: name
    value: Maria
  remember_user_facts (call_SizWddyvhKv6e5DSElBdCkI5)
 Call ID: call_SizWddyvhKv6e5DSElBdCkI5
  Args:
    key: location
    value: Sofia
  remember_user_facts (call_UbmWPhDiLffGb0LxIxAbviWj)
 Call ID: call_UbmWPhDiLffGb0LxIxAbviWj
  Args:
    key: hobby
    value: hiking
====================